# Genre-Conditioned Album Cover Generation - Conditional VAE

MSAI 495 project. Trains a cVAE on 20 genres of album covers and generates new ones conditioned on a chosen genre.

## 1. Data loading

In [2]:
# Colab setup - mount drive, unzip dataset, install extra packages
from google.colab import drive
drive.mount('/content/drive')

!cp '/content/drive/MyDrive/GAID.zip' /content/GAID.zip
!unzip -q /content/GAID.zip -d /content/

!pip -q install keras-tuner gradio

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
DATA_DIR = '/content/GAID'
IMG_SIZE = 128
BATCH_SIZE = 512

genres = sorted(os.listdir(DATA_DIR))
genres = [g for g in genres if os.path.isdir(os.path.join(DATA_DIR, g))]
num_genres = len(genres)
genre_to_idx = {g: i for i, g in enumerate(genres)}

print('num genres:', num_genres)
print(genres)

In [ ]:
class AlbumCoverDataset(Dataset):
    def __init__(self, root, genres, img_size=128):
        self.samples = []
        for g in genres:
            folder = os.path.join(root, g)
            for fname in os.listdir(folder):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(folder, fname), genre_to_idx[g]))

        # Decode every image once into a single tensor in CPU RAM.
        # 20k x 3 x 128 x 128 float32 ~ 3.9 GB, fits easily on Colab.
        # This eliminates the per-batch JPEG decode overhead.
        tfm = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
        ])
        self.images = torch.empty(len(self.samples), 3, img_size, img_size, dtype=torch.float32)
        self.labels = torch.empty(len(self.samples), dtype=torch.long)
        for i, (path, lbl) in enumerate(self.samples):
            with Image.open(path) as im:
                self.images[i] = tfm(im.convert('RGB'))
            self.labels[i] = lbl
            if (i + 1) % 2000 == 0:
                print(f'  decoded {i + 1}/{len(self.samples)}')

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], int(self.labels[idx])

dataset = AlbumCoverDataset(DATA_DIR, genres, img_size=IMG_SIZE)
print('total images:', len(dataset))
print('images tensor:', dataset.images.shape, dataset.images.dtype)

In [ ]:
# Train / val split
val_size = int(0.1 * len(dataset))
train_size = len(dataset) - val_size
train_set, val_set = random_split(dataset, [train_size, val_size],
                                  generator=torch.Generator().manual_seed(SEED))

# Data is already in RAM, so workers add overhead - use 0.
# pin_memory speeds up the host -> GPU copy.
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=True)

print('train:', train_size, '| val:', val_size)

In [ ]:
# Quick check - count per genre
labels = [lbl for _, lbl in dataset.samples]
counts = np.bincount(labels, minlength=num_genres)
for g, c in zip(genres, counts):
    print(f'{g:20s} {c}')

In [1]:
# Show a few sample covers from each genre
fig, axes = plt.subplots(num_genres, 5, figsize=(8, num_genres * 1.5))
for r, g in enumerate(genres):
    g_idxs = [i for i, (_, lbl) in enumerate(dataset.samples) if lbl == r]
    picks = random.sample(g_idxs, 5)
    for c, i in enumerate(picks):
        img, _ = dataset[i]
        axes[r, c].imshow(img.permute(1, 2, 0).numpy())
        axes[r, c].axis('off')
        if c == 0:
            axes[r, c].set_title(g, fontsize=8, loc='left')
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

## 2. cVAE model

Encoder: 4 strided conv blocks downsampling 128 -> 64 -> 32 -> 16 -> 8.
Decoder: mirror with transposed convs back to 128.
Genre is one-hot encoded and concatenated to the flattened encoder features (for mu/logvar) and to z (for the decoder).

In [ ]:
LATENT_DIM = 128
BASE_CH = 64

def conv_block(in_c, out_c):
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, kernel_size=4, stride=2, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True),
    )

def deconv_block(in_c, out_c):
    return nn.Sequential(
        nn.ConvTranspose2d(in_c, out_c, kernel_size=4, stride=2, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True),
    )

class CVAE(nn.Module):
    def __init__(self, num_genres, latent_dim=128, base_ch=64, img_size=128):
        super().__init__()
        self.num_genres = num_genres
        self.latent_dim = latent_dim
        self.base_ch = base_ch
        # 128 -> 64 -> 32 -> 16 -> 8
        self.enc = nn.Sequential(
            conv_block(3, base_ch),
            conv_block(base_ch, base_ch * 2),
            conv_block(base_ch * 2, base_ch * 4),
            conv_block(base_ch * 4, base_ch * 8),
        )
        self.feat_size = img_size // 16  # 8 for 128x128
        self.flat_dim = base_ch * 8 * self.feat_size * self.feat_size

        self.fc_mu = nn.Linear(self.flat_dim + num_genres, latent_dim)
        self.fc_logvar = nn.Linear(self.flat_dim + num_genres, latent_dim)

        self.fc_dec = nn.Linear(latent_dim + num_genres, self.flat_dim)
        # 8 -> 16 -> 32 -> 64 -> 128
        self.dec = nn.Sequential(
            deconv_block(base_ch * 8, base_ch * 4),
            deconv_block(base_ch * 4, base_ch * 2),
            deconv_block(base_ch * 2, base_ch),
            nn.ConvTranspose2d(base_ch, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def one_hot(self, y):
        return F.one_hot(y, num_classes=self.num_genres).float()

    def encode(self, x, y_oh):
        h = self.enc(x).flatten(1)
        h = torch.cat([h, y_oh], dim=1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, y_oh):
        h = self.fc_dec(torch.cat([z, y_oh], dim=1))
        h = h.view(-1, self.base_ch * 8, self.feat_size, self.feat_size)
        return self.dec(h)

    def forward(self, x, y):
        y_oh = self.one_hot(y)
        mu, logvar = self.encode(x, y_oh)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z, y_oh)
        return x_hat, mu, logvar

In [ ]:
# quick sanity check on shapes
model = CVAE(num_genres=num_genres, latent_dim=LATENT_DIM, base_ch=BASE_CH, img_size=IMG_SIZE).to(device)
x = torch.randn(4, 3, IMG_SIZE, IMG_SIZE).to(device)
y = torch.randint(0, num_genres, (4,)).to(device)
x_hat, mu, logvar = model(x, y)
print('x_hat:', x_hat.shape, 'mu:', mu.shape, 'logvar:', logvar.shape)
print('params:', sum(p.numel() for p in model.parameters()) / 1e6, 'M')

## 3. Training

ELBO loss = reconstruction (BCE summed over pixels) + beta * KL divergence.
We train with Adam, save the best checkpoint by val loss, and plot the curves.

In [ ]:
def vae_loss(x, x_hat, mu, logvar, beta=1.0):
    # sum over pixels, mean over batch
    recon = F.binary_cross_entropy(x_hat, x, reduction='sum') / x.size(0)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return recon + beta * kl, recon, kl

In [ ]:
def train_one_epoch(model, loader, optimizer, beta):
    model.train()
    total, total_recon, total_kl, n = 0.0, 0.0, 0.0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        x_hat, mu, logvar = model(x, y)
        loss, recon, kl = vae_loss(x, x_hat, mu, logvar, beta=beta)
        loss.backward()
        optimizer.step()
        total += loss.item(); total_recon += recon.item(); total_kl += kl.item(); n += 1
    return total / n, total_recon / n, total_kl / n

@torch.no_grad()
def evaluate(model, loader, beta):
    model.eval()
    total, total_recon, total_kl, n = 0.0, 0.0, 0.0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
        x_hat, mu, logvar = model(x, y)
        loss, recon, kl = vae_loss(x, x_hat, mu, logvar, beta=beta)
        total += loss.item(); total_recon += recon.item(); total_kl += kl.item(); n += 1
    return total / n, total_recon / n, total_kl / n

In [ ]:
EPOCHS = 30
LR = 1e-3
BETA = 1.0

model = CVAE(num_genres=num_genres, latent_dim=LATENT_DIM, base_ch=BASE_CH, img_size=IMG_SIZE).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

os.makedirs('checkpoints', exist_ok=True)
best_val = float('inf')
history = {'train': [], 'val': [], 'recon': [], 'kl': []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_recon, tr_kl = train_one_epoch(model, train_loader, optimizer, BETA)
    val_loss, val_recon, val_kl = evaluate(model, val_loader, BETA)
    history['train'].append(tr_loss)
    history['val'].append(val_loss)
    history['recon'].append(val_recon)
    history['kl'].append(val_kl)
    print(f'epoch {epoch:02d} | train {tr_loss:.2f} | val {val_loss:.2f} (recon {val_recon:.2f}, kl {val_kl:.2f})')
    if val_loss < best_val:
        best_val = val_loss
        torch.save({'model': model.state_dict(),
                    'num_genres': num_genres,
                    'latent_dim': LATENT_DIM,
                    'base_ch': BASE_CH,
                    'img_size': IMG_SIZE,
                    'genres': genres},
                   'checkpoints/best.pt')

print('best val:', best_val)

In [ ]:
# Loss curves
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(history['train'], label='train')
ax[0].plot(history['val'], label='val')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('total loss'); ax[0].legend()
ax[1].plot(history['recon'], label='val recon')
ax[1].plot(history['kl'], label='val kl')
ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.show()

## 4. Latent space exploration

We look at:
1. Reconstructions on the validation set.
2. Samples from N(0, I) for each genre (fix z, vary genre).
3. Latent interpolation between two random z's at a fixed genre.
4. Genre interpolation - fix z and blend two one-hot genre vectors.

In [ ]:
# Load best checkpoint
ckpt = torch.load('checkpoints/best.pt', map_location=device)
model = CVAE(num_genres=ckpt['num_genres'], latent_dim=ckpt['latent_dim'],
             base_ch=ckpt['base_ch'], img_size=ckpt['img_size']).to(device)
model.load_state_dict(ckpt['model'])
model.eval()
print('loaded best.pt')

In [ ]:
# 1. Reconstruction grid - top row real, bottom row reconstructed
x_batch, y_batch = next(iter(val_loader))
x_batch, y_batch = x_batch[:8].to(device), y_batch[:8].to(device)

with torch.no_grad():
    x_hat, _, _ = model(x_batch, y_batch)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(x_batch[i].cpu().permute(1, 2, 0).numpy())
    axes[0, i].axis('off')
    axes[1, i].imshow(x_hat[i].cpu().permute(1, 2, 0).numpy())
    axes[1, i].axis('off')
axes[0, 0].set_title('real', loc='left')
axes[1, 0].set_title('recon', loc='left')
plt.tight_layout(); plt.show()

In [ ]:
# 2. Per-genre samples - fix z, vary the genre label
torch.manual_seed(0)
z_fixed = torch.randn(1, model.latent_dim, device=device)

fig, axes = plt.subplots(4, 5, figsize=(10, 8))
with torch.no_grad():
    for i, g in enumerate(genres):
        y_oh = F.one_hot(torch.tensor([i], device=device), num_classes=num_genres).float()
        img = model.decode(z_fixed, y_oh)[0].cpu().permute(1, 2, 0).numpy()
        ax = axes[i // 5, i % 5]
        ax.imshow(img)
        ax.set_title(g, fontsize=9)
        ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 3. Latent interpolation - between two random z's, fixed genre
target_genre = 'Jazz'
g_idx = genre_to_idx[target_genre]
y_oh = F.one_hot(torch.tensor([g_idx], device=device), num_classes=num_genres).float()

torch.manual_seed(1)
z_a = torch.randn(1, model.latent_dim, device=device)
z_b = torch.randn(1, model.latent_dim, device=device)

steps = 8
fig, axes = plt.subplots(1, steps, figsize=(steps * 1.5, 2))
with torch.no_grad():
    for k, t in enumerate(np.linspace(0, 1, steps)):
        z = (1 - t) * z_a + t * z_b
        img = model.decode(z, y_oh)[0].cpu().permute(1, 2, 0).numpy()
        axes[k].imshow(img); axes[k].axis('off')
        axes[k].set_title(f't={t:.2f}', fontsize=8)
fig.suptitle(f'latent interpolation - genre: {target_genre}', y=1.05)
plt.tight_layout(); plt.show()

In [ ]:
# 4. Genre interpolation - fix z, blend two one-hot genre vectors
g_a, g_b = 'Classical', 'DeathMetal'
ia, ib = genre_to_idx[g_a], genre_to_idx[g_b]

torch.manual_seed(2)
z = torch.randn(1, model.latent_dim, device=device)

steps = 8
fig, axes = plt.subplots(1, steps, figsize=(steps * 1.5, 2))
with torch.no_grad():
    for k, t in enumerate(np.linspace(0, 1, steps)):
        oh = torch.zeros(1, num_genres, device=device)
        oh[0, ia] = 1 - t
        oh[0, ib] = t
        img = model.decode(z, oh)[0].cpu().permute(1, 2, 0).numpy()
        axes[k].imshow(img); axes[k].axis('off')
        axes[k].set_title(f'{1-t:.1f}{g_a[:3]} / {t:.1f}{g_b[:3]}', fontsize=7)
fig.suptitle(f'genre interpolation: {g_a} -> {g_b}', y=1.05)
plt.tight_layout(); plt.show()

## 5. Hyperparameter tuning - Hyperband

Use keras-tuner's Hyperband algorithm to search over latent_dim, base_ch, learning rate and beta. We subclass `kt.Tuner` and override `run_trial` to train our PyTorch cVAE for the number of epochs Hyperband assigns. To keep tuning fast, each trial uses a subset of the training data.

In [ ]:
import keras_tuner as kt
from torch.utils.data import Subset

# Use a subset of training data for tuning to keep it fast
tune_size = min(4000, len(train_set))
tune_indices = np.random.RandomState(SEED).choice(len(train_set), tune_size, replace=False)
tune_train = Subset(train_set, tune_indices.tolist())

tune_train_loader = DataLoader(tune_train, batch_size=512, shuffle=True,
                               num_workers=0, pin_memory=True, drop_last=True)
tune_val_loader = DataLoader(val_set, batch_size=512, shuffle=False,
                             num_workers=0, pin_memory=True)
print('tune train size:', len(tune_train))

In [ ]:
class CVAETuner(kt.Tuner):
    def run_trial(self, trial, train_loader, val_loader):
        hp = trial.hyperparameters
        latent_dim = hp.Choice('latent_dim', [64, 128, 256])
        base_ch = hp.Choice('base_ch', [32, 64])
        lr = hp.Float('lr', 1e-4, 3e-3, sampling='log')
        beta = hp.Choice('beta', [0.5, 1.0, 2.0])
        epochs = hp.get('tuner/epochs')

        m = CVAE(num_genres=num_genres, latent_dim=latent_dim,
                 base_ch=base_ch, img_size=IMG_SIZE).to(device)
        opt = torch.optim.Adam(m.parameters(), lr=lr)

        best = float('inf')
        for ep in range(epochs):
            train_one_epoch(m, train_loader, opt, beta)
            val, _, _ = evaluate(m, val_loader, beta)
            if val < best:
                best = val
        return {'val_loss': best}

oracle = kt.oracles.HyperbandOracle(
    objective=kt.Objective('val_loss', 'min'),
    max_epochs=8,
    factor=3,
    hyperband_iterations=1,
)

tuner = CVAETuner(
    oracle=oracle,
    hypermodel=lambda hp: None,
    directory='kt_dir',
    project_name='cvae',
    overwrite=True,
)
tuner.search_space_summary()

In [ ]:
tuner.search(tune_train_loader, tune_val_loader)
tuner.results_summary(num_trials=5)

In [ ]:
# Retrain the best config on the full training set and save it
best_hp = tuner.get_best_hyperparameters(1)[0]
print('best hp:', best_hp.values)

best_latent = best_hp.get('latent_dim')
best_base = best_hp.get('base_ch')
best_lr = best_hp.get('lr')
best_beta = best_hp.get('beta')

model = CVAE(num_genres=num_genres, latent_dim=best_latent,
             base_ch=best_base, img_size=IMG_SIZE).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=best_lr)

EPOCHS_FINAL = 30
best_val = float('inf')
for epoch in range(1, EPOCHS_FINAL + 1):
    tr_loss, _, _ = train_one_epoch(model, train_loader, optimizer, best_beta)
    val_loss, _, _ = evaluate(model, val_loader, best_beta)
    print(f'epoch {epoch:02d} | train {tr_loss:.2f} | val {val_loss:.2f}')
    if val_loss < best_val:
        best_val = val_loss
        torch.save({'model': model.state_dict(),
                    'num_genres': num_genres,
                    'latent_dim': best_latent,
                    'base_ch': best_base,
                    'img_size': IMG_SIZE,
                    'genres': genres,
                    'beta': best_beta,
                    'lr': best_lr},
                   'checkpoints/best_tuned.pt')

print('best tuned val:', best_val)

## 6. Gradio GUI

Simple interactive UI with two tabs:
- Generate: pick a genre and number of samples, get back generated covers.
- Interpolate: pick two genres and a blend ratio.

In [ ]:
# Load the best tuned checkpoint for the GUI
ckpt = torch.load('checkpoints/best_tuned.pt', map_location=device)
model = CVAE(num_genres=ckpt['num_genres'], latent_dim=ckpt['latent_dim'],
             base_ch=ckpt['base_ch'], img_size=ckpt['img_size']).to(device)
model.load_state_dict(ckpt['model'])
model.eval()
genres = ckpt['genres']
print('loaded best_tuned.pt')

In [ ]:
import gradio as gr

def generate(genre, n_samples, temperature):
    g_idx = genres.index(genre)
    y_oh = F.one_hot(torch.tensor([g_idx] * n_samples, device=device),
                     num_classes=len(genres)).float()
    z = torch.randn(n_samples, model.latent_dim, device=device) * temperature
    with torch.no_grad():
        imgs = model.decode(z, y_oh).cpu()
    out = []
    for i in range(n_samples):
        arr = (imgs[i].permute(1, 2, 0).numpy() * 255).clip(0, 255).astype(np.uint8)
        out.append(arr)
    return out

def interpolate(genre_a, genre_b, t):
    ia, ib = genres.index(genre_a), genres.index(genre_b)
    oh = torch.zeros(1, len(genres), device=device)
    oh[0, ia] = 1 - t
    oh[0, ib] = t
    z = torch.randn(1, model.latent_dim, device=device)
    with torch.no_grad():
        img = model.decode(z, oh)[0].cpu()
    arr = (img.permute(1, 2, 0).numpy() * 255).clip(0, 255).astype(np.uint8)
    return arr

with gr.Blocks(title='Album Cover cVAE') as demo:
    gr.Markdown('# Genre-Conditioned Album Cover Generator')

    with gr.Tab('Generate'):
        with gr.Row():
            g_dd = gr.Dropdown(genres, value=genres[0], label='Genre')
            n_sl = gr.Slider(1, 8, value=4, step=1, label='Number of samples')
            t_sl = gr.Slider(0.1, 2.0, value=1.0, step=0.1, label='Temperature')
        gen_btn = gr.Button('Generate')
        gen_out = gr.Gallery(label='Generated covers', columns=4)
        gen_btn.click(generate, [g_dd, n_sl, t_sl], gen_out)

    with gr.Tab('Interpolate'):
        with gr.Row():
            ga_dd = gr.Dropdown(genres, value=genres[0], label='Genre A')
            gb_dd = gr.Dropdown(genres, value=genres[1], label='Genre B')
            t_in = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label='Blend (A -> B)')
        int_btn = gr.Button('Generate')
        int_out = gr.Image(label='Interpolated cover')
        int_btn.click(interpolate, [ga_dd, gb_dd, t_in], int_out)

demo.launch(share=True, debug=False)